# MeetMind ASR Benchmark — Tamil + English

Benchmark **Sarvam Saaras v4** against **OpenAI Whisper large-v3** for MeetMind-style audio:
Tamil, English, Tamil-English code-mixing, and meeting speech.

**Important:** use human-verified reference transcripts in `metadata.csv`. Do not use another ASR output as ground truth.

Expected ZIP structure:

```text
meetmind_benchmark/
├── audio/
│   ├── tamil_01.wav
│   ├── english_01.wav
│   ├── codemix_01.wav
│   └── meeting_01.wav
└── metadata.csv
```

`metadata.csv` columns:
`id,audio,category,reference`

Categories:
`tamil`, `english`, `codemix`, `meeting`


In [ ]:
!pip -q install -U transformers accelerate librosa soundfile jiwer pandas numpy tqdm sarvamai


In [ ]:
import os, re, time, subprocess, warnings
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from jiwer import wer, cer
warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Upload your benchmark ZIP

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_path = next(iter(uploaded.keys()))
print("Uploaded:", zip_path)


In [ ]:
import zipfile

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

candidates = [
    "/content/meetmind_benchmark",
    "/content/meetmind_benchmark/meetmind_benchmark",
]
DATASET_DIR = next((p for p in candidates if os.path.isdir(p)), None)

if DATASET_DIR is None:
    raise FileNotFoundError(
        "Could not find /content/meetmind_benchmark after extraction."
    )

AUDIO_DIR = os.path.join(DATASET_DIR, "audio")
METADATA_PATH = os.path.join(DATASET_DIR, "metadata.csv")
NORMALIZED_DIR = "/content/meetmind_normalized"
os.makedirs(NORMALIZED_DIR, exist_ok=True)

print("Dataset:", DATASET_DIR)
print("Audio:", AUDIO_DIR)
print("Metadata:", METADATA_PATH)


In [ ]:
df = pd.read_csv(METADATA_PATH)

required = {"id", "audio", "category", "reference"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

allowed = {"tamil", "english", "codemix", "meeting"}
bad_categories = set(df["category"].dropna().str.lower()) - allowed
if bad_categories:
    raise ValueError(f"Unsupported categories: {bad_categories}")

print("Samples:", len(df))
display(df["category"].value_counts().rename_axis("category").to_frame("count"))
display(df.head())


## 2. Normalize all audio to 16 kHz mono WAV

In [ ]:
missing_files = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    rel = str(row["audio"])
    src = rel if os.path.isabs(rel) else os.path.join(DATASET_DIR, rel)

    if not os.path.exists(src):
        alt = os.path.join(AUDIO_DIR, os.path.basename(rel))
        src = alt if os.path.exists(alt) else src

    if not os.path.exists(src):
        missing_files.append((row["id"], src))
        continue

    out = os.path.join(NORMALIZED_DIR, f"{row['id']}.wav")

    subprocess.run(
        [
            "ffmpeg", "-y", "-i", src,
            "-ar", "16000",
            "-ac", "1",
            "-sample_fmt", "s16",
            out
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )

if missing_files:
    print("Missing files:")
    for item in missing_files:
        print(item)
else:
    print("All audio normalized successfully.")


## 3. Load Whisper large-v3

In [ ]:
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

WHISPER_ID = "openai/whisper-large-v3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(WHISPER_ID)

whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    WHISPER_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
    use_safetensors=True
).to(DEVICE)

whisper_model.eval()
print("Whisper loaded on", DEVICE)


In [ ]:
import librosa

def transcribe_whisper(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        DEVICE,
        dtype=DTYPE
    )

    with torch.inference_mode():
        generated_ids = whisper_model.generate(
            input_features,
            task="transcribe",
            num_beams=5
        )

    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()


## 4. Configure Saaras v4

In [ ]:
import getpass
from sarvamai import SarvamAI

os.environ["SARVAM_API_KEY"] = getpass.getpass(
    "Enter your Sarvam API key: "
)

sarvam = SarvamAI(
    api_subscription_key=os.environ["SARVAM_API_KEY"]
)

def transcribe_saaras(audio_path):
    with open(audio_path, "rb") as f:
        response = sarvam.speech_to_text.transcribe(
            file=f,
            model="saaras:v4",
            language_code="unknown"
        )

    transcript = getattr(response, "transcript", None)

    if transcript is None and isinstance(response, dict):
        transcript = response.get("transcript", "")

    if transcript is None:
        transcript = str(response)

    return str(transcript).strip()


## 5. Run both ASR systems

In [ ]:
results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Benchmarking"):
    audio_path = os.path.join(
        NORMALIZED_DIR,
        f"{row['id']}.wav"
    )

    if not os.path.exists(audio_path):
        continue

    t0 = time.perf_counter()
    try:
        whisper_text = transcribe_whisper(audio_path)
        whisper_error = ""
    except Exception as e:
        whisper_text = ""
        whisper_error = repr(e)
    whisper_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    try:
        saaras_text = transcribe_saaras(audio_path)
        saaras_error = ""
    except Exception as e:
        saaras_text = ""
        saaras_error = repr(e)
    saaras_time = time.perf_counter() - t0

    results.append({
        "id": row["id"],
        "category": str(row["category"]).lower(),
        "reference": str(row["reference"]),
        "whisper": whisper_text,
        "saaras": saaras_text,
        "whisper_time_sec": whisper_time,
        "saaras_time_sec": saaras_time,
        "whisper_error": whisper_error,
        "saaras_error": saaras_error,
    })

results_df = pd.DataFrame(results)

print("Completed:", len(results_df), "samples")
display(results_df.head())


## 6. WER/CER evaluation

In [ ]:
def safe_metric(fn, ref, hyp):
    try:
        return float(fn(str(ref), str(hyp)))
    except Exception:
        return np.nan

for model in ["whisper", "saaras"]:
    results_df[f"{model}_WER"] = results_df.apply(
        lambda r: safe_metric(
            wer, r["reference"], r[model]
        ),
        axis=1
    )

    results_df[f"{model}_CER"] = results_df.apply(
        lambda r: safe_metric(
            cer, r["reference"], r[model]
        ),
        axis=1
    )

summary = pd.DataFrame({
    "Model": ["Whisper large-v3", "Saaras v4"],
    "Mean WER": [
        results_df["whisper_WER"].mean(),
        results_df["saaras_WER"].mean()
    ],
    "Mean CER": [
        results_df["whisper_CER"].mean(),
        results_df["saaras_CER"].mean()
    ],
    "Mean latency (sec)": [
        results_df["whisper_time_sec"].mean(),
        results_df["saaras_time_sec"].mean()
    ]
})

display(summary)


## 7. Per-category benchmark

In [ ]:
rows = []

for category, g in results_df.groupby("category"):
    rows.append({
        "Category": category,
        "Whisper WER": g["whisper_WER"].mean(),
        "Saaras WER": g["saaras_WER"].mean(),
        "Whisper CER": g["whisper_CER"].mean(),
        "Saaras CER": g["saaras_CER"].mean(),
        "Whisper latency sec": g["whisper_time_sec"].mean(),
        "Saaras latency sec": g["saaras_time_sec"].mean(),
        "Samples": len(g)
    })

category_df = pd.DataFrame(rows).sort_values("Category")
display(category_df)


## 8. Tamil/English script preservation

In [ ]:
def tamil_char_ratio(text):
    text = str(text)
    return len(re.findall(r"[\u0B80-\u0BFF]", text)) / max(len(text), 1)

def english_char_ratio(text):
    text = str(text)
    return len(re.findall(r"[A-Za-z]", text)) / max(len(text), 1)

for col in ["reference", "whisper", "saaras"]:
    results_df[f"{col}_tamil_char_ratio"] = (
        results_df[col].apply(tamil_char_ratio)
    )
    results_df[f"{col}_english_char_ratio"] = (
        results_df[col].apply(english_char_ratio)
    )

codemix = results_df[
    results_df["category"] == "codemix"
].copy()

if len(codemix):
    script_summary = pd.DataFrame({
        "Metric": [
            "Reference Tamil chars",
            "Reference English chars",
            "Whisper Tamil chars",
            "Whisper English chars",
            "Saaras Tamil chars",
            "Saaras English chars"
        ],
        "Mean ratio": [
            codemix["reference_tamil_char_ratio"].mean(),
            codemix["reference_english_char_ratio"].mean(),
            codemix["whisper_tamil_char_ratio"].mean(),
            codemix["whisper_english_char_ratio"].mean(),
            codemix["saaras_tamil_char_ratio"].mean(),
            codemix["saaras_english_char_ratio"].mean()
        ]
    })
    display(script_summary)
else:
    print("No codemix samples found.")


## 9. Side-by-side transcription review

In [ ]:
review_cols = [
    "id",
    "category",
    "reference",
    "whisper",
    "saaras",
    "whisper_WER",
    "saaras_WER"
]

display(results_df[review_cols])


## 10. Export benchmark results

In [ ]:
RESULTS_PATH = "/content/meetmind_asr_results.csv"
CATEGORY_PATH = "/content/meetmind_asr_category_results.csv"
SUMMARY_PATH = "/content/meetmind_asr_summary.csv"

results_df.to_csv(
    RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_df.to_csv(
    CATEGORY_PATH,
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Created:")
print(RESULTS_PATH)
print(CATEGORY_PATH)
print(SUMMARY_PATH)


In [ ]:
from google.colab import files

files.download(RESULTS_PATH)
files.download(CATEGORY_PATH)
files.download(SUMMARY_PATH)


## How to choose the MeetMind winner

Do not select the winner from overall WER alone.

Prioritize:

1. **Tamil-English code-mixed WER/CER**
2. **Meeting-category WER/CER**
3. Tamil and English script preservation
4. Real-time latency and stability
5. Human review of technical words, names, interruptions, and natural Indian speech

Lower WER/CER is better.

The latency comparison is not infrastructure-equivalent because Saaras is API-based while Whisper runs locally on the Colab GPU. Use latency as an engineering signal, not the sole model-quality metric.

For production MeetMind, the next stage after this benchmark should test the winning model on the actual WebSocket microphone pipeline and then evaluate final meeting transcription + speaker diarization.
